Poisson Equation
===

We solve the Poisson equation $-\Delta u = f$ on the unit square, with homogeneous Dirichlet boundary conditions.

Import Netgen/NGSolve Python modules:

In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw

The unit_square is a predefined domain, use Netgen to generate a mesh:

In [ ]:
mesh = Mesh(unit_square.GenerateMesh(maxh=0.2))
Draw (mesh);

The boundary parts of the mesh have names:

In [ ]:
print (mesh.GetBoundaries())

Define a finite element space on that mesh, and get trial- and test-functions:

In [ ]:
fes = H1(mesh, order=3)
u,v = fes.TnT()
print ("ndof =", fes.ndof)

The variational formulation is: find $u$ such that $u = 0$ on the boundary, and

$$
\int \nabla u \nabla v \, dx = \int f v \, dx
$$

for all test-functions $v$. We write down this equation, give Dirichlet conditions on boundary regions, and solve it:

In [ ]:
f = 32*(y*(1-y)+x*(1-x))

gfu = Solve(grad(u)*grad(v)*dx == f*v*dx,
            u[BND("left|right|bottom|top")] == 0)

Plot the solution:

In [ ]:
Draw (gfu, mesh);

In [ ]:
Draw (-grad(gfu), mesh, "Flux", vectors= { "grid_size" : 40});

Calculate error:

In [ ]:
exact = 16*x*(1-x)*y*(1-y)
print ("L2-error:", sqrt(Integrate((gfu-exact)**2, mesh)))

Behind the scenes
---

`Solve` sets up a bilinear-form and a linear-form

$$
a(u,v) = \int \nabla u \nabla v
\qquad \text{and} \qquad
f(v) = \int f v,
$$

assembles matrix and vector, and calls a sparse direct solver. All these steps are available one by one. Now the Dirichlet boundary is a property of the finite element space:

In [ ]:
fes = H1(mesh, order=3, dirichlet="left|right|bottom|top")
u,v = fes.TnT()

a = BilinearForm(grad(u)*grad(v)*dx).Assemble()
lf = LinearForm(f*v*dx).Assemble()

In [ ]:
print(lf.vec)
print(a.mat)

In [ ]:
gfu2 = GridFunction(fes)
gfu2.vec.data = a.mat.Inverse(fes.FreeDofs()) * lf.vec

print ("difference:", sqrt(Integrate((gfu-gfu2)**2, mesh)))

This is where the [i-tutorials](../unit-1.1-poisson/poisson.ipynb) continue.